In this notebook, the points representing comprehensive schools in the study area are retrieved from the Lounaistieto WFS.

In [ ]:
import geopandas as gpd
import osmnx as ox
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"
NOTEBOOK_PATH

In [ ]:
import geopandas
import pandas as pd
# Lounaistieto WFS
LOUNAISTIETO_WFS_BASE_URL = (
    "https://geoserver.lounaistieto.fi/geoserver/service_points/ows"
    "?service=wfs"
    "&version=2.0.0"
    "&request=GetFeature"
    "&srsName=EPSG:3067"
    "&typeName={layer:s}"
)
# Import layer of educational and learning service points in Southwest Finland
kasvatus_ja_koulutuspalvelut = (
    geopandas.read_file(
        LOUNAISTIETO_WFS_BASE_URL.format(layer="kasvatus_ja_koulutuspalvelut")
    )
    .set_crs("EPSG: 3067")
)
#Select points only inside study area
aluerajaus = gpd.read_file(DATA_DIRECTORY/"aluerajaus"/"aluerajaus.shp")
kasvatus_ja_koulutuspalvelut = kasvatus_ja_koulutuspalvelut[kasvatus_ja_koulutuspalvelut.within(aluerajaus.at[0,"geometry"])]
# Select only schools
koulut = kasvatus_ja_koulutuspalvelut[kasvatus_ja_koulutuspalvelut["palvelukohdetyyppi"]=="Koulut (perusopetus)"]
koulut.head()

In [ ]:
# There are some points that do not provide education but are other facilities such as gyms. 
# Let's filter in only the educational (perusopetus) facilies
koulut = koulut[koulut["palvelut"].str.contains("perusopetus", na=False)].copy()
koulut.shape

In [ ]:
# Some rows have same point location but different school unit (yksikkö). 
# These don't need to be in separate points so we can aggregate them together.
# Here we group schools by their coordinates and combine them together, keeping all names.
koulut["geometry_id"] = koulut.geometry.to_wkt()
schools_combined = koulut.dissolve(
    by="geometry_id",
    aggfunc={
        "kohteen_nimi": lambda x: ", ".join(x),
        "osoite": "first",
        "palvelut": "first"
    },
    as_index=False
).sort_values("osoite")

In [ ]:
schools_combined.shape

In [ ]:
schools_combined.to_file(OUTPUT_DIRECTORY / "turku_schools.gpkg")